# 课程选课智能助手 · 实现与测试记录

本 Notebook 记录项目的实际实现与运行结果，代码全部位于项目 Python 文件中，Notebook 只做调用与验证，避免出现两套实现：

| 文件 | 作用 |
|---|---|
| `course_mcp_server.py` | 课程业务 MCP Server（4 个工具，SQLite 事务与业务规则） |
| `course_engine.py` | 模型、Prompt、RAG、Skill、两个 Agent、LangGraph 工作流 |
| `web_search.py` | Tavily MCP 接入与失败回退 |
| `app.py` / `web/index.html` | FastAPI 接口与中文对话页面 |

启动服务：`.venv\Scripts\python.exe app.py`，浏览器访问 http://127.0.0.1:8765 。


In [1]:
from pathlib import Path
import sys, json, sqlite3, asyncio, time

ROOT = Path.cwd()
if not (ROOT / 'course_engine.py').exists() and (ROOT / '综合大项目' / 'course_engine.py').exists():
    ROOT = ROOT / '综合大项目'
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT))

print('项目目录：', ROOT)
print('Python：', sys.version.split()[0], sys.executable)
print('数据库：', (ROOT / 'data' / 'course_selection.sqlite3').is_file())
print('课程文档：', [p.name for p in sorted((ROOT / 'knowledge' / 'courses').glob('*.md'))])

项目目录： C:\Users\Administrator\Desktop\综合大项目
Python： 3.12.12 C:\Users\Administrator\Desktop\综合大项目\.venv\Scripts\python.exe
数据库： True
课程文档： ['C001.md', 'C002.md', 'C003.md', 'C004.md', 'C005.md', '课程目录.md']


## 1. 需求分析与数据库

三张表：`users`（演示身份）、`courses`（课程、容量、教室、软硬件）、`enrollments`（选课记录，`user_id + course_id` 唯一）。
业务规则：每门课 20 人、每人 2 门、禁止重复、选退课需确认、写库前事务内再次校验。
初始数据：C004 已满（20/20），其余课程 0/20。

In [2]:
db_path = ROOT / 'data' / 'course_selection.sqlite3'
with sqlite3.connect(f'file:{db_path}?mode=ro', uri=True) as conn:
    for table in ('users', 'courses', 'enrollments'):
        ddl = conn.execute("SELECT sql FROM sqlite_master WHERE name=?", (table,)).fetchone()[0]
        count = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
        print(f'== {table}（{count} 行）')
        print(ddl.replace('\n', ' ')[:260], '\n')
    print('触发器：', [row[0] for row in conn.execute("SELECT name FROM sqlite_master WHERE type='trigger'")])

== users（23 行）
CREATE TABLE users (user_id TEXT PRIMARY KEY, name TEXT NOT NULL, role TEXT NOT NULL CHECK(role IN ('student','teacher'))) 

== courses（5 行）
CREATE TABLE courses (course_id TEXT PRIMARY KEY, name TEXT NOT NULL UNIQUE, capacity INTEGER NOT NULL CHECK(capacity>0), equipment TEXT NOT NULL, software TEXT NOT NULL, classroom TEXT NOT NULL DEFAULT '待安排') 

== enrollments（20 行）
CREATE TABLE enrollments (enrollment_id INTEGER PRIMARY KEY, user_id TEXT NOT NULL REFERENCES users(user_id), course_id TEXT NOT NULL REFERENCES courses(course_id), selected_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP, UNIQUE(user_id,course_id)) 

触发器： ['trg_course_capacity', 'trg_user_limit']


## 2. Model 与 Prompt

- 模型：`deepseek-flash`（`temperature=0.5`），通过 `.env` 的 `DEEPSEEK_API_KEY` 调用；
- Prompt 版本：`PROMPT_V0`（简单）与 `PROMPT_V1`（项目使用），`reports/prompt_compare.md` 保存了同一问题的对比结果。

In [3]:
import course_engine as engine

model = engine.create_model()
print('模型：', engine.CHAT_MODEL, '| 温度：', engine.TEMPERATURE)
print('\nPROMPT_V1 摘要：')
print('\n'.join(engine.PROMPT_V1.strip().splitlines()[:8]))

answer = model.invoke('用一句话说明课程咨询助手的职责，不要超过 40 字。').content
print('\n模型自述：', answer)

模型： deepseek-flash | 温度： 0.5

PROMPT_V1 摘要：
你是“校园课程助手”，服务对象是准备选课的大学生，回答必须简洁、准确、可执行。

【任务】
1. 回答课程定位、教学目标、先修要求、学时、模块、实验环境、软硬件要求、考核方式等问题。
2. 学生同时问多门课时，按课程分别说明再比较，不要混在一起。
3. 支持追问：结合“最近对话”判断“这门课”“它”指的是哪门课。

【工具规则】



模型自述： 解答课程疑问，匹配学员需求，提供选课建议并协助报名。


## 3. RAG 课程知识库

- 资料：`knowledge/courses/课程目录.md` + `C001`—`C005` 五份课程文档；
- 切分：`MarkdownHeaderTextSplitter` 按章节 + `RecursiveCharacterTextSplitter`（420 字，重叠 60）；
- 向量：通义 `text-embedding-v1`（带本地缓存 `data/embedding_cache.json`）；
- 存储与检索：`InMemoryVectorStore` + 按课程编号过滤。

In [4]:
retriever = engine.build_retriever()
print('知识库统计：', json.dumps(retriever.stats(), ensure_ascii=False))

hits = retriever.search('C002 的先修要求是什么', k=3, course_id='C002')
for doc in hits:
    print(f"- {doc.metadata['source']} · {doc.metadata['section']}：{doc.page_content[:80].replace(chr(10), ' ')}")

知识库统计：

 {"documents": 45, "courses": ["ALL", "C001", "C002", "C003", "C004", "C005"], "sources": ["C001.md", "C002.md", "C003.md", "C004.md", "C005.md", "课程目录.md"]}


- C002.md · 先修要求与适合人群：## 先修要求与适合人群   需要 Python 编程基础（可参考 C001 Python 数据分析实践），以及线性代数、概率统计基础。适合对预测、模型、算法原
- C002.md · 课程定位与概览：# C002 机器学习入门   ## 课程定位与概览   课程方向：算法建模。难度：★★★★。软件门槛：中；个人硬件门槛：低，GPU 可选。建议年级：大二或大三
- C002.md · 考核方式与学习收获：## 考核方式与学习收获   考核：平时 20% + 实验 40% + 项目 40%。完成课程后，学生能独立完成中小规模建模任务，理解模型评估的基本方法与局限。


In [5]:
result = await engine.chat('C002 机器学习入门主要学什么？', 'nb-knowledge', 'S001', use_web=False)
print(result['reply'][:600])

C002 机器学习入门共 48 学时，重点讲授经典机器学习算法的原理、使用和评估，核心是判断“什么场景适合什么算法、模型效果好不好”，**不涉及**大模型训练与工程部署。

- **8 个模块**：数学回顾、数据预处理、回归、分类、集成学习、无监督学习、评估与调参、综合项目
- **实验**：用 scikit-learn 和 numpy 做预处理流水线、房价预测、分类对比、模型融合、客户分群、调参、端到端项目
- **先修**：Python 基础（可参考 C001）和线代、概率统计；建议在大二或大三修读，难度 ★★★★
- **硬件**：个人硬件门槛低，GPU 可选

来源：C002.md · 课程定位与概览；课程模块与实验安排；教学目标


## 4. 课程业务 MCP

`course_mcp_server.py` 以 stdio 启动，注册 `list_courses`、`list_enrollments`、`select_course`、`drop_course`。
`course_engine.CourseMCPClient` 通过 `mcp.client.stdio` 建立会话，工具返回的 JSON 直接进入业务回复。

In [6]:
import course_engine as engine

client = await engine.connect_course_mcp()
print('工具列表：')
for tool in await client.list_tools():
    print(' -', tool['name'], '：', tool['description'])

courses = await client.call('list_courses')
print('\n课程目录：')
for course in courses['courses']:
    print(f"  {course['course_id']} {course['name']} | {course['classroom']} | {course['enrolled']}/{course['capacity']} 剩余 {course['remaining']}")

print('\n我的选课（S001）：', json.dumps(await client.call('list_enrollments', {'user_id': 'S001'}), ensure_ascii=False)[:200])

工具列表：
 - list_courses ： 查询课程目录：课程名称、教室、已选人数与剩余名额。
 - list_enrollments ： 查询指定用户当前已选课程、教室与选课时间。
 - select_course ： 选课（需用户确认后调用）。事务内校验课程存在、重复、满员与个人上限。
 - drop_course ： 退课（需用户确认后调用）：删除本人选课记录并释放名额。

课程目录：
  C001 Python 数据分析实践 | 教学楼 A 座 301（数据分析实验室） | 0/20 剩余 20
  C002 机器学习入门 | 教学楼 A 座 301（数据分析实验室） | 0/20 剩余 20
  C003 物联网传感器实践 | 实验楼 B 座 201（物联网创新实验室） | 0/20 剩余 20
  C004 数字媒体设计基础 | 实验楼 B 座 202（数字媒体实验室） | 20/20 剩余 0
  C005 网络基础与安全实验 | 实训楼 C 座 101（网络技术实验室） | 0/20 剩余 20

我的选课（S001）： {"ok": true, "user_id": "S001", "name": "测试用户001", "limit": 2, "count": 0, "courses": []}


## 5. LangGraph 工作流

节点：`route → knowledge / recommend / catalog / mine / plan → confirm（interrupt）→ commit / reject`。
`InMemorySaver` 以 `thread_id=session_id` 保存会话；确认前不写库，确认后由 `commit` 节点调用 MCP 写入。

In [7]:
graph = await engine.get_runtime()
print('节点：', [n for n in graph.get_graph().nodes])
print('\nMermaid 结构：')
print(graph.get_graph().draw_mermaid()[:1200])

节点： ['__start__', 'route', 'knowledge', 'recommend', 'catalog', 'mine', 'plan', 'confirm', 'commit', 'reject', 'clarify', '__end__']

Mermaid 结构：
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	route(route)
	knowledge(knowledge)
	recommend(recommend)
	catalog(catalog)
	mine(mine)
	plan(plan)
	confirm(confirm)
	commit(commit)
	reject(reject)
	clarify(clarify)
	__end__([<p>__end__</p>]):::last
	__start__ --> route;
	confirm -.-> commit;
	confirm -.-> reject;
	plan -.-> __end__;
	plan -.-> confirm;
	route -.-> catalog;
	route -.-> clarify;
	route -.-> knowledge;
	route -.-> mine;
	route -.-> plan;
	route -.-> recommend;
	catalog --> __end__;
	clarify --> __end__;
	commit --> __end__;
	knowledge --> __end__;
	mine --> __end__;
	recommend --> __end__;
	reject --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [8]:
# 选课 → 确认 → 我的选课 → 退课（使用演示用户 S003，结束后清理）
with sqlite3.connect(ROOT / 'data' / 'course_selection.sqlite3') as conn:
    conn.execute("DELETE FROM enrollments WHERE user_id = 'S003'")
    conn.commit()

for message in ('我想选 C001', '确认', '我的选课', '帮我退掉 C001', '确认'):
    out = await engine.chat(message, 'nb-flow', 'S003', use_web=False)
    print(f"\n> {message}")
    print('  waiting =', out['waiting_for_confirmation'], '| pending =', out['pending_action'])
    print(' ', out['reply'].replace('\n', '\n   ')[:400])


> 我想选 C001


  waiting = True | pending = select
  将为你选择：C001 Python 数据分析实践
   上课教室：教学楼 A 座 301（数据分析实验室）
   当前名额：0/20，剩余 20 个
   你已选 0/2 门
   
   点击“确认选课”按钮或回复“确认”才会写入数据库；回复“取消”不做任何修改。

> 确认
  waiting = False | pending = None
  选课成功：C001 Python 数据分析实践。
   当前 1/20，剩余 19 个名额；你已选 1 门。
   （业务数据由课程 MCP 工具直接查询数据库生成，未经模型改写，保证与数据库一致。）

> 我的选课
  waiting = False | pending = None
  S003 测试用户003 已选 1/2 门：
     - C001 Python 数据分析实践（教学楼 A 座 301（数据分析实验室），2026-11-05 06:57:48）
   
   如需更换课程，可以先退课再选新课，两项操作都需要确认。
   （业务数据由课程 MCP 工具直接查询数据库生成，未经模型改写，保证与数据库一致。）



> 帮我退掉 C001
  waiting = True | pending = drop
  将为你退掉：C001 Python 数据分析实践
   退课后该课程名额会从 1/20 释放 1 个
   
   点击“确认退课”按钮或回复“确认”才会删除选课记录；回复“取消”不做任何修改。

> 确认
  waiting = False | pending = None
  退课成功：C001 Python 数据分析实践，名额已释放。
   当前 0/20，剩余 20 个名额；你已选 0 门。
   （业务数据由课程 MCP 工具直接查询数据库生成，未经模型改写，保证与数据库一致。）


## 6. Skill 与多 Agent

- **课程推荐 Skill**（`skills/course_recommendation/SKILL.md`）：触发条件（比较/推荐/方向）、输入、五维比较步骤、输出格式与边界；
- **课程咨询 Agent**：`create_agent` + `search_courses` 工具，负责知识问答与比较；
- **选课业务 Agent**：封装课程 MCP 工具，负责目录、我的选课、选退课预检与事务提交，回复中的数字直接来自数据库。

In [9]:
print(engine.load_recommendation_skill()[:700])
print('\n选课业务 Agent：', engine.SelectionAgent.__doc__.strip().splitlines()[0])

# Skill：课程比较与推荐（course_recommendation）

## 触发条件
学生消息中出现下列任一情况时启用本技能：
- 对比多门课程：「C001 和 C002 有什么区别」「比较一下数据分析与机器学习」；
- 请求推荐：「我该选哪门课」「推荐一门适合我的课」「我对做网站/做硬件感兴趣，选什么」；
- 询问方向与门槛：「哪门课更适合零基础」「哪门硬件要求高」。

## 输入
| 输入 | 说明 |
|---|---|
| 学生原话 | 兴趣方向、基础、每周可投入时间等线索 |
| 候选课程编号 | 消息中出现的 C001—C005；未出现时默认比较全部五门 |
| 课程资料片段 | 由 RAG 对每门课分别检索（课程定位、先修、模块、实验环境） |
| 数据库实时名额 | 由课程 MCP 的 `list_courses` 提供，用于判断是否已满 |

## 执行步骤
1. 识别候选课程，逐门课独立检索，不混用不同课程的内容。
2. 按五个维度并列比较：课程方向、难度（★数）、先修要求、软硬件门槛、适合人群。
3. 结合学生已表达的兴趣给出 1—2 门首选，并说明推荐依据（引用课程资料，不编造学时或教师信息）。
4. 若首选课程已满（`remaining = 0`），推荐方向相近且仍有名额的课程，并说明差异。
5. 资料不足时明确写「课程资料中未提及」，不补充猜测内容。
6. 结尾询问是否需要直接选课；选课必须走确认流程，技能本身不写数据库。

## 输出格式
1. 一句结论（推荐哪门、为什么）；
2. 3—5 行对比要点（每行一个维度，课程编号开头）；
3.

选课业务 Agent： 选课业务 Agent。


## 7. 测试结果

- 端到端 15 个场景：`reports/test_results.md`（课程咨询、追问、比较、名额、满员、取消/确认、重复、上限、退课、无答案、Tavily 回退、持久化、我的选课、状态回归）；
- MCP 与并发抢最后名额：`reports/mcp_and_concurrency.md`；
- Prompt 对比：`reports/prompt_compare.md`。

In [10]:
summary = (ROOT / 'reports' / 'test_results.md').read_text(encoding='utf-8').splitlines()
print('\n'.join(summary[:6]))
print('...')
print('\n'.join([line for line in summary if line.startswith('- 测试')][:6]))

# 课程选课智能助手 · 端到端测试记录

- 生成时间：2026-11-05 14:56:10
- 结果：15/15 通过

| 编号 | 测试场景 | 输入 | 预期结果 | 实际结果 | 是否通过 |
...
- 测试 1：全部校验通过；耗时 3.6s
- 测试 2：全部校验通过；耗时 3.4s
- 测试 3：全部校验通过；耗时 7.3s
- 测试 4：全部校验通过；耗时 0.0s
- 测试 5：全部校验通过；备注：C004 操作前剩余 0，操作后剩余 0；耗时 0.0s
- 测试 6：全部校验通过；耗时 0.0s


## 8. 小结

1. RAG 保证课程介绍有依据（资料未提及会明确说明），MCP + SQLite 保证教室、名额与选退课结果真实；
2. LangGraph 的 `interrupt` / `Command(resume)` 让写操作必须经过人工确认，取消时数据库不变；
3. 事务 + 触发器 + 写前复核解决了并发抢最后名额的问题（并发测试：2 人抢 1 个名额，仅 1 人成功）；
4. 不足：会话记忆存在内存中，服务重启后丢失；演示身份未接入真实教务认证；Tavily 密钥未配置时只能回退本地知识。